In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score
import os

# 1. Load the Historical Data
data_path = os.path.join('..', 'data', 'processed', 'historical_solar_data.csv')
print(f"Loading data from {data_path}...")
df = pd.read_csv(data_path)

# Convert timestamp back to datetime objects
df['timestamp'] = pd.to_datetime(df['timestamp'])

# 2. Feature Engineering (Helping the AI understand time)
# The model can't read "2025-05-01 14:00:00", so we break it into numbers
df['hour'] = df['timestamp'].dt.hour
df['month'] = df['timestamp'].dt.month
df['day_of_year'] = df['timestamp'].dt.dayofyear

# Define our inputs (X) and what we want to predict (y)
features = ['temperature_c', 'humidity_percent', 'wind_speed_kmh', 'cloud_cover_percent', 'hour', 'month', 'day_of_year']
target = 'power_output_kw'

X = df[features]
y = df[target]

# 3. Train/Test Split
# We give the model 80% of the data to learn from, and hide 20% to test it later
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training on {len(X_train)} rows, Testing on {len(X_test)} rows.\n")

# 4. Initialize and Train the XGBoost Model
print("Initializing XGBoost Engine...")
model = xgb.XGBRegressor(
    n_estimators=500,     # Build 500 decision trees
    learning_rate=0.05,   # Learn steadily
    max_depth=6,          # How complex each tree can get
    random_state=42
)

print("Training in progress (this may take a few seconds)...")
model.fit(X_train, y_train)

# 5. Evaluate the AI
print("\n--- Model Evaluation ---")
predictions = model.predict(X_test)

# Calculate metrics
mae = mean_absolute_error(y_test, predictions)
r2 = r2_score(y_test, predictions)

print(f"Mean Absolute Error (MAE): {mae:.2f} kW")
print(f"R-Squared (Accuracy Score): {r2 * 100:.2f}%")

# Save the trained model for production use
os.makedirs(os.path.join('..', 'models'), exist_ok=True)
model_path = os.path.join('..', 'models', 'solar_forecaster.json')
model.save_model(model_path)
print(f"\nModel successfully saved to {model_path}!")

Loading data from ../data/processed/historical_solar_data.csv...
Training on 7008 rows, Testing on 1753 rows.

Initializing XGBoost Engine...
Training in progress (this may take a few seconds)...

--- Model Evaluation ---
Mean Absolute Error (MAE): 2.86 kW
R-Squared (Accuracy Score): 99.82%

Model successfully saved to ../models/solar_forecaster.json!
